# KMV (2018) earnings process — step-1 test notebook

Walks through the whole step-1 pipeline and checks each piece:

1. environment & imports
2. **validate** the simulator against KMV's published parameters (`simulate.py`, `discretize.py`)
3. build target moments from the GRID data (`grid_loader.py`)
4. estimate the process by SMM (`estimate.py`)
5. produce the Table-2 table + export the 33-state Markov chain (`run.py`, `discretize.py`)
6. (optional) compare optimisers and inspect the exported chain

Run the cells top to bottom. Each section says what it checks and what a good result looks like.

> **Launch this notebook from the folder that contains the `kmv_earnings/` package**
> (the same place `python -m kmv_earnings.run` works). Cell 1.1 checks this and fixes the path if it can.

## 1. Environment & imports

In [13]:
# 1.1  Make sure we're in the folder that contains the kmv_earnings package.
import os, sys
from pathlib import Path

here = Path.cwd()
if not (here / "kmv_earnings").is_dir():
    for cand in [here] + list(here.parents) + list(here.glob("**/kmv_earnings")):
        base = cand.parent if cand.name == "kmv_earnings" else cand
        if (base / "kmv_earnings").is_dir():
            os.chdir(base); break

print("Working directory:", Path.cwd())
assert (Path.cwd() / "kmv_earnings").is_dir(), \
    "Could not find the kmv_earnings package. Move this notebook next to it."
print("Found kmv_earnings package [ok]")

Working directory: c:\Users\kacpe\Desktop\makro research\magister HANK\kmv_grid_step1
Found kmv_earnings package [ok]


In [14]:
# 1.2  Check dependencies (install the two optional ones if missing).
import importlib, subprocess
def ensure(pkg):
    try:
        importlib.import_module(pkg); print(f"{pkg:10s} [ok]")
    except ImportError:
        print(f"{pkg:10s} missing -> installing ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
for p in ["numpy", "scipy", "pandas", "jinja2"]:   # jinja2 only for LaTeX export
    ensure(p)

numpy      [ok]
scipy      [ok]
pandas     [ok]
jinja2     [ok]


In [15]:
# 1.3  Import the package modules. If this runs, everything is wired up.
import importlib
import kmv_earnings.simulate as sim
import kmv_earnings.discretize as disc
import kmv_earnings.estimate as est
import kmv_earnings.grid_loader as grid
import kmv_earnings.run as run
for m in (sim, disc, est, grid, run):
    importlib.reload(m)   # pick up edits without restarting the kernel
print("All modules imported [ok]")
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

All modules imported [ok]


## 2. Validate the simulator

Takes KMV's *published* Table 3 parameters, runs them through the simulator, and
compares the moments we get with KMV's own model column. No fitting here — it's a
correctness check of the engine (parameters -> moments, forward direction).

**Good result:** *Model Estimated* close to
`0.70 / 0.23 / 0.46 / 16.5 / 12.1 / 0.56 / 0.67 / 0.85`.
Variances and fractions very close; kurtosis a little low (sample-design + grid).

In [16]:
params_kmv = sim.KMV_TABLE3_PARAMS
print("KMV Table 3 parameters (quarterly rates):")
for k, v in params_kmv.items():
    print(f"  {k:8s} = {v}")

KMV Table 3 parameters (quarterly rates):
  lambda1  = 0.08
  beta1    = 0.761
  sigma1   = 1.74
  lambda2  = 0.007
  beta2    = 0.009
  sigma2   = 1.53


In [17]:
# Simulate at KMV's parameters and build the 3-column table.
sim_kwargs = dict(n_workers=50_000, n_years_keep=36)   # raise n_workers for smoother numbers
table_validate, _ = run.build_table(sim.KMV_US_DATA_TARGETS, params_kmv, sim_kwargs)
table_validate

,Moment,Data,Model Estimated,Model Discretized
0,Variance: annual log earns,0.700,0.688,0.748
1,Variance: 1yr change,0.230,0.227,0.270
2,Variance: 5yr change,0.460,0.503,0.570
3,Kurtosis: 1yr change,17.800,14.554,9.574
4,Kurtosis: 5yr change,11.600,10.884,8.384
5,Frac 1yr change < 10%,0.540,0.547,0.684
6,Frac 1yr change < 20%,0.710,0.659,0.696
7,Frac 1yr change < 50%,0.860,0.840,0.795


In [18]:
# See the Monte-Carlo noise shrink as n_workers grows.
for n in (10_000, 50_000, 150_000):
    m = sim.model_moments(params_kmv, n_workers=n, n_years_keep=36)
    print(f"n_workers={n:>7}: var_log={m['var_log_earns']:.3f}  "
          f"kurt_d1={m['kurt_d1']:.2f}  frac<10%={m['frac_d1_lt_10']:.3f}")

n_workers=  10000: var_log=0.690  kurt_d1=14.32  frac<10%=0.548
n_workers=  50000: var_log=0.688  kurt_d1=14.55  frac<10%=0.547
n_workers= 150000: var_log=0.692  kurt_d1=14.55  frac<10%=0.547


## 3. Build target moments from the GRID data (`grid_loader.py`)

Reads the GRID `Stats_*.csv` and turns the reported statistics into the 8 target
moments per country. The "fraction of small changes" moments aren't in GRID
directly — they're interpolated from the 1-year change percentile grid.



In [22]:
# Locate the GRID csv (searches the repo if the default path isn't there).
csv_path = Path("data/Stats_20260720145608.csv")
if not csv_path.exists():
    hits = list(Path.cwd().glob("**/Stats_*.csv"))
    csv_path = hits[0] if hits else csv_path
print("Using GRID file:", csv_path, "[ok]" if csv_path.exists() else "<- NOT FOUND")

Using GRID file: data\Stats_20260720145608.csv [ok]


In [23]:
# Peek at the raw file.
raw = pd.read_csv(csv_path)
print("countries:", raw['country'].unique())
print("years    :", raw['year'].min(), "-", raw['year'].max())
print("cells    :", raw[['gender','age']].drop_duplicates().to_dict('records'))
raw.head(3)

countries: <StringArray>
['FRA', 'GER', 'USA']
Length: 3, dtype: str
years    : 1991 - 2019
cells    : [{'gender': 'Male', 'age': '25-55'}]


,country,year,gender,age,nobs_log_inc,std_log_inc,nobs_res_1yr_log_chg,std_res_1yr_log_chg,kurt_res_1yr_log_chg,p1_res_1yr_log_chg,...,p97_5_res_1yr_log_chg,p99_res_1yr_log_chg,p99_9_res_1yr_log_chg,p99_99_res_1yr_log_chg,p9010_res_1yr_log_chg,p9050_res_1yr_log_chg,p5010_res_1yr_log_chg,nobs_res_5yr_log_chg,std_res_5yr_log_chg,kurt_res_5yr_log_chg
0,FRA,1991,Male,25-55,302370.000,0.712,273094.000,0.476,17.324,-1.942,...,1.024,1.561,2.450,3.153,0.477,0.213,0.264,222436.000,0.591,12.567
1,FRA,1992,Male,25-55,322639.000,0.709,276553.000,0.472,16.690,-1.911,...,0.999,1.515,2.481,3.047,0.491,0.205,0.286,233395.000,0.588,12.281
2,FRA,1993,Male,25-55,314540.000,0.699,234599.000,0.496,16.692,-2.064,...,1.002,1.494,2.411,3.102,0.490,0.196,0.294,241255.000,0.555,13.484


Time period different compared to KMV - newer data and less data from past (1998-2019 vs KMV 1978-2013)

In [25]:
d=pd.read_csv('data/Stats_20260720145608.csv'); print(d[d.country=='USA']['year'].agg(['min','max']))

min    1998
max    2019
Name: year, dtype: int64


In [24]:
# Build and save targets for each country.
os.makedirs("targets", exist_ok=True)
targets = {}
for c in ["USA", "FRA", "GER"]:
    t = grid.targets_from_grid_stats_csv(str(csv_path), c)
    grid.save_targets(t, f"targets/{c.lower()}_grid.json")
    targets[c] = t

cmp = pd.DataFrame({m: {c: targets[c][m] for c in targets} for m in sim.MOMENT_ORDER}).T
cmp.insert(0, "KMV_US", [sim.KMV_US_DATA_TARGETS[m] for m in sim.MOMENT_ORDER])
cmp.index = [sim.MOMENT_LABELS[m] for m in sim.MOMENT_ORDER]
cmp

,KMV_US,USA,FRA,GER
Variance: annual log earns,0.700,0.945,0.488,0.645
Variance: 1yr change,0.230,0.323,0.206,0.148
Variance: 5yr change,0.460,0.618,0.336,0.285
Kurtosis: 1yr change,17.800,12.865,15.864,17.803
Kurtosis: 5yr change,11.600,8.828,11.682,11.210
Frac 1yr change < 10%,0.540,0.387,0.528,0.611
Frac 1yr change < 20%,0.710,0.587,0.725,0.788
Frac 1yr change < 50%,0.860,0.811,0.864,0.894


## 4. Estimate the process by SMM (`estimate.py`)

Searches for the 6 parameters whose simulated moments best match a country's
targets (moments -> parameters, inverse direction). Global optimiser =
differential evolution, then Nelder-Mead polish; `x0` warm-starts from KMV.

This is the slow cell. Settings below are **quick** (confirms it runs). For
thesis numbers use the "serious" settings in the comment.

In [36]:
COUNTRY = "USA"          # GER/FRA/USA
tgt = targets[COUNTRY]

# quick (~1-3 min):
sim_kwargs_est = dict(n_workers=10_000, n_years_keep=36, steps_per_quarter=3)
maxiter, popsize = 15, 6

# serious (slower; uncomment for real runs):
#sim_kwargs_est = dict(n_workers=50_000, n_years_keep=36, steps_per_quarter=6)
# maxiter, popsize = 100, 12

params_hat, res = est.estimate(
    tgt, sim_kwargs=sim_kwargs_est, maxiter=maxiter, popsize=popsize,
    workers=1,                     # keep 1 on Windows for notebook stability
    x0=sim.KMV_TABLE3_PARAMS,      # warm start
    disp=True,
)
print("\nEstimated parameters (quarterly rates):")
for k, v in params_hat.items():
    print(f"  {k:8s} = {v:.4f}")
print(f"final objective f = {res.fun:.5f}")

differential_evolution step 1: f(x)= 0.3291072828917191
differential_evolution step 2: f(x)= 0.2862146747721989
differential_evolution step 3: f(x)= 0.2862146747721989
differential_evolution step 4: f(x)= 0.08973059729806974
differential_evolution step 5: f(x)= 0.08973059729806974
differential_evolution step 6: f(x)= 0.08973059729806974
differential_evolution step 7: f(x)= 0.08973059729806974
differential_evolution step 8: f(x)= 0.08973059729806974
differential_evolution step 9: f(x)= 0.08973059729806974
differential_evolution step 10: f(x)= 0.08973059729806974
differential_evolution step 11: f(x)= 0.08973059729806974
differential_evolution step 12: f(x)= 0.08973059729806974
differential_evolution step 13: f(x)= 0.0637076621710883
differential_evolution step 14: f(x)= 0.04824015917713463
differential_evolution step 15: f(x)= 0.04824015917713463

Estimated parameters (quarterly rates):
  lambda1  = 0.1464
  beta1    = 1.5589
  sigma1   = 1.7546
  lambda2  = 0.0104
  beta2    = 0.0084
  

## 5. Table + export the Markov chain (`run.py`, `discretize.py`)

Rebuilds the 3-column table at the estimated parameters and writes the outputs —
the `.csv`/`.tex` table and the four `income_process_*.txt` files that ARE the
33-state quarterly Markov chain handed to step 2.

In [37]:
out_dir = f"output/{COUNTRY.lower()}_nb"
os.makedirs(out_dir, exist_ok=True)

table_fit, disc_obj = run.build_table(tgt, params_hat,
                                      dict(n_workers=50_000, n_years_keep=36))
display(table_fit)

table_fit.to_csv(f"{out_dir}/table2.csv", index=False)
try:
    with open(f"{out_dir}/table2.tex", "w") as f:
        f.write(table_fit.to_latex(index=False, float_format="%.2f",
                caption="Earnings Process Estimation Fit", label="tab:earnings_fit"))
except Exception as e:
    print("(skipped .tex:", e, ")")

disc.export_chain(disc_obj, out_dir)
est.save_params(params_hat, f"{out_dir}/params.json")
print("\nWrote to", out_dir, ":")
for p in sorted(os.listdir(out_dir)):
    print("  ", p)

,Moment,Data,Model Estimated,Model Discretized
0,Variance: annual log earns,0.945,0.907,0.917
1,Variance: 1yr change,0.323,0.252,0.261
2,Variance: 5yr change,0.618,0.577,0.600
3,Kurtosis: 1yr change,12.865,12.806,8.606
4,Kurtosis: 5yr change,8.828,9.673,7.831
5,Frac 1yr change < 10%,0.387,0.447,0.622
6,Frac 1yr change < 20%,0.587,0.604,0.636
7,Frac 1yr change < 50%,0.811,0.829,0.778



Wrote to output/usa_nb :
   income_process_P.txt
   income_process_grid.txt
   income_process_pi.txt
   income_process_zgrid.txt
   params.json
   table2.csv
   table2.tex


In [38]:
# Inspect the exported chain — what step 2 will consume.
e  = np.loadtxt(f"{out_dir}/income_process_grid.txt")   # earnings levels (mean 1)
P  = np.loadtxt(f"{out_dir}/income_process_P.txt")      # quarterly transition matrix
pi = np.loadtxt(f"{out_dir}/income_process_pi.txt")     # stationary distribution
print(f"states: {len(e)}   (should be 33)")
print(f"transition matrix shape: {P.shape}")
print(f"rows of P sum to 1? {np.allclose(P.sum(1), 1)}")
print(f"mean earnings under stationary dist: {pi @ e:.4f}  (should be ~1)")
logm = pi @ np.log(e)
print(f"implied var(log e): {pi @ (np.log(e)-logm)**2:.3f}")

states: 33   (should be 33)
transition matrix shape: (33, 33)
rows of P sum to 1? True
mean earnings under stationary dist: 1.0000  (should be ~1)
implied var(log e): 1.559


## 6. (optional) Compare optimisers — differential evolution vs TikTak

Runs both global optimisers with a small budget, then re-scores each solution
**out-of-sample** on a big panel with a fresh seed. Watch whether they agree,
especially on the *persistent* component (`lambda2`,`beta2`,`sigma2`) — short
budgets can leave it weakly identified. Skip if `tiktak.py` isn't in your copy.

In [39]:
try:
    import kmv_earnings.tiktak as tk
    importlib.reload(tk); have_tiktak = True
except Exception as e:
    have_tiktak = False; print("tiktak not available:", e)

In [40]:
if have_tiktak:
    sk = dict(n_workers=10_000, n_years_keep=36, steps_per_quarter=3)
    p_de, _ = est.estimate(tgt, sim_kwargs=sk, maxiter=15, popsize=6,
                           workers=1, x0=sim.KMV_TABLE3_PARAMS, disp=False)
    p_tt, info = tk.tiktak(tgt, sim_kwargs=sk, n_sobol=128, n_local=6, disp=False)

    hp = dict(n_workers=150_000, n_years_keep=36, steps_per_quarter=6, seed=999)
    t = np.array([tgt[m] for m in sim.MOMENT_ORDER])
    def score(p):
        m = sim.model_moments(p, **hp)
        v = np.array([m[k] for k in sim.MOMENT_ORDER])
        return float(np.sum(((v - t)/t)**2))

    dfp = pd.DataFrame({"DE": p_de, "TikTak": p_tt}).loc[list(sim.PARAM_ORDER)]
    dfp.loc["obj (out-of-sample)"] = [score(p_de), score(p_tt)]
    display(dfp)
    print("Do lambda2/beta2/sigma2 agree? If not, run longer budgets before trusting them.")

,DE,TikTak
lambda1,0.146,0.146
beta1,1.559,1.681
sigma1,1.755,1.797
lambda2,0.010,0.012
beta2,0.008,0.007
sigma2,1.451,1.301
obj (out-of-sample),0.085,0.093


Do lambda2/beta2/sigma2 agree? If not, run longer budgets before trusting them.


---
### Recap — a fully working step 1

- **§2 validate:** Model Estimated ~ KMV's model column (kurtosis a touch low).
- **§3 targets:** three JSONs written;
- **§4 estimate:** objective falls to a small value; parameters economically
  sensible (persistent shocks rare & large, transitory frequent & small).
- **§5 export:** `income_process_*.txt` written; P rows sum to 1; mean earnings ~ 1.
  These files are the hand-off to step 2 (the HANK / sequence-space model).

**For thesis-grade numbers:** switch §4 to "serious" settings and confirm both optimisers agree in §6.